# Modèle XGBoost 

## Chargements des bibliothèques et des données

Dans un premier temps, importons notre jeu de données. Prenons garde à utiliser le dataset d'entraînement pour ne pas fausser les prévisions. 

Le dataframe est un tableau de 21 colonnes et 17147 lignes. La target est le prix du logement. Il contient des variables continues et des variables discrètes.["prix","id","date","zipcode","lat",'long']

In [7]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import SelectFromModel
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score
import xgboost as xgb
import matplotlib.pyplot as plt

df = pd.read_csv("train_data.csv")  

## Feature engineering

In [5]:
# Calcul du prix moyen par zipcode
df["prix_moy_zipcode"] = df.groupby("zipcode")["prix"].transform("mean")

df['zipcode'] = df['zipcode'].astype(str)
n_clusters = 3  
kmeans = KMeans(n_clusters=n_clusters, random_state=0).fit(df[['lat', 'long']])
df['cluster'] = kmeans.labels_
# Calcul du prix moyen par cluster
df['prix_moyen_par_cluster'] = df['cluster'].map(df.groupby('cluster')['prix'].mean())

# Pour les variables explicatives, j'ai supprimé toutes celles jugées difficiles à traiter 
X = df.drop(columns=["prix","id","date","zipcode","lat",'long',
       'm2_jardin', 'm2_soussol'])  # Variables explicatives
y = df["prix"]  # Target

# Train/test avec 20% pour le test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Modèle avec validation croisée

In [6]:
# Pipeline amélioré
pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler()),
    ("feature_selection", SelectFromModel(RandomForestRegressor(n_estimators=100, random_state=42))),
    ("model", xgb.XGBRegressor(objective="reg:squarederror", n_estimators=200, learning_rate=0.1))
])

# Optimisation des hyperparamètres
param_grid = {
    "model__n_estimators": [100, 200, 500],
    "model__max_depth": [3, 5, 7],
    "model__learning_rate": [0.01, 0.1, 0.2]
}

grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring="neg_root_mean_squared_error", n_jobs=-1)
grid_search.fit(X_train, y_train)

# Meilleur modèle et prédictions
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

# Évaluation
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"Meilleurs paramètres : {grid_search.best_params_}")
print(f"RMSE optimisé : {rmse:.2f}")
print(f"R² optimisé : {r2:.3f}")

Meilleurs paramètres : {'model__learning_rate': 0.2, 'model__max_depth': 3, 'model__n_estimators': 200}
RMSE optimisé : 163133.21
R² optimisé : 0.807


## Code pour soumission Kaggle

In [6]:
import pandas as pd

# Charger les données de test
df_test = pd.read_csv("test_data.csv")

# Charger les données d'entraînement pour récupérer les moyennes par zipcode
df_train = pd.read_csv("train_data.csv")
prix_moy_zipcode = df_train.groupby("zipcode")["prix"].mean()

# Ajouter la colonne prix moyen par zipcode dans df_test (basé sur df_train)
df_test["prix_moy_zipcode"] = df_test["zipcode"].map(prix_moy_zipcode)

# Remplacer les NaN (au cas où certains zipcodes de test ne sont pas dans train)
df_test["prix_moy_zipcode"].fillna(df_train["prix"].mean(), inplace=True)

# Supprimer les colonnes non utilisées dans la prédiction
X_test_final = df_test.drop(columns=["id", "date", "zipcode", "lat", "long"])

# Prédiction avec le meilleur modèle entraîné
y_pred_test = best_model.predict(X_test_final)

# Créer un DataFrame pour soumission Kaggle
submission = pd.DataFrame({"id": df_test["id"], "prix": y_pred_test})

# Sauvegarder en CSV
submission.to_csv("submission.csv", index=False)

print("📂 Fichier 'submission.csv' généré avec succès !")




📂 Fichier 'submission.csv' généré avec succès !


C:\Users\qevan\AppData\Local\Temp\ipykernel_23700\196488081.py:14: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_test["prix_moy_zipcode"].fillna(df_train["prix"].mean(), inplace=True)
